# Solar Filament v3 — free GPU, reuse trained v2 weights

**Purpose:** improve the **existing 0.23 model** without paying for another full training run. This experiment performs a wider foreground/area threshold search, uses a disjoint confirmation subset, reports PQ + a local instance-Dice proxy + split/merge counts, and compares 4-view TTA.

**Before running:** In Kaggle **Add Input**, attach (1) official `filament-segmentation-2026` competition dataset and (2) the **output** of your successful `notebookbde8e3d22c` run containing `solar_artifacts/best.pt`. Set Accelerator to GPU and Internet On. Run **Save Version → Save & Run All**. Do not attach external test annotations.

**Important:** This is NOT EdgeAttNet training. It reuses the already trained v2 checkpoint. Confirmation PQ and Dice proxy are not official Kaggle scores; leaderboard score is known only after submission. If v3 fails to improve, retain the 0.23 submission.

In [ ]:
import sys, subprocess, pathlib, shutil, json, torch
ROOT = pathlib.Path('/kaggle/working/Solar-Filament-Segmentation')
assert torch.cuda.is_available(), 'Enable Kaggle GPU first.'
assert not ROOT.exists(), 'Start a fresh session (or use the existing cloned project).'
subprocess.run(['git','clone','--depth','1','https://github.com/ahpa5246-lgtm/Solar-Filament-Segmentation.git',str(ROOT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pycocotools==2.0.8'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(ROOT)],check=True)
sys.path.insert(0,str(ROOT))
from solar.data import discover
print('GPU:',torch.cuda.get_device_name(0))
print('Competition data:',discover('/kaggle/input')[0])
candidates = sorted(pathlib.Path('/kaggle/input').rglob('best.pt'))
print('Found pretrained checkpoints:',[str(p) for p in candidates])
assert len(candidates)==1, 'Attach the successful v2 notebook OUTPUT via Add Input, containing exactly one best.pt.'
OUT=pathlib.Path('/kaggle/working/v3_artifacts')
OUT.mkdir(exist_ok=True)
shutil.copy2(candidates[0],OUT/'best.pt')
print('Reusing trained checkpoint:',OUT/'best.pt')

## 1 — Run regression tests before expensive inference
All tests use synthetic data; no test labels are accessed.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests')],check=True)

## 2 — Threshold search and disjoint confirmation
24 parameter settings plus v2 reference, 24 tuning observations, 12 distinct confirmation observations. Four-view TTA is used **both** in tuning and final inference.

In [ ]:
BATCH=2
TTA='d4'  # Four views: identity, horizontal/vertical flips, 180° rotation
subprocess.run([sys.executable,'-m','solar.tune','--data','/kaggle/input',
                '--checkpoint',str(OUT/'best.pt'),'--out',str(OUT),
                '--profile','precision','--max-images','24','--audit-images','12',
                '--tta-mode',TTA,'--batch',str(BATCH),'--seed','42'],check=True)
report=json.loads((OUT/'confirmation_results.json').read_text())
print('Confirmation (not Kaggle):',json.dumps(report['results'],indent=2))
print('Selected parameters:',report['selected_params'])

## 3 — Full test prediction and RLE validation
The test set has 180 H-alpha images. This step uses only the images, never any test ground truth.

In [ ]:
subprocess.run([sys.executable,'-m','solar.predict','--data','/kaggle/input',
                '--checkpoint',str(OUT/'best.pt'),'--params',str(OUT/'best_params.json'),
                '--out','/kaggle/working/submission.csv','--batch',str(BATCH),
                '--tta-mode',TTA],check=True)
import pandas as pd
s=pd.read_csv('/kaggle/working/submission.csv',dtype=str)
assert list(s.columns)==['filament_id','segmentation_rle']
assert s.filament_id.is_unique and s.segmentation_rle.notna().all()
print('Predicted instances:',len(s))
print('Observations with detections:',s.filament_id.str.rsplit('_',n=1).str[0].nunique())
print('Ready:',pathlib.Path('/kaggle/working/submission.csv'))

## 4 — Save evidence
Keep v2 and v3 submissions separate; only submit v3 if its independent confirmation is credible. Never replace the original trained checkpoint.

In [ ]:
subprocess.run([sys.executable,'-m','pip','freeze'],stdout=open(OUT/'environment_frozen.txt','w'),check=True)
print('Artifacts:',[p.name for p in OUT.iterdir()])
print('IMPORTANT: Public Kaggle score is unknown until you submit the new CSV.')